In [1]:
from scripts.reqs import *

/Users/boriskov/Documents/Sch21/Classic ML from scratch/ML5_Decision_trees.ID_1254804-1/ml5.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv('data/train.csv')
data.shape

(72983, 34)

In [3]:
data = data.drop(columns=['PRIMEUNIT', 'AUCGUART'])
data.PurchDate = pd.to_datetime(data.PurchDate)

def split_by_three_date(data : pd.DataFrame, date_col : str):
    df = data.copy()
    l = len(data)
    df = df.sort_values(date_col)
    
    train_end = l // 3
    val_end = (2 * l) // 3

    train = df.iloc[:train_end]
    val = df.iloc[train_end:val_end]
    test = df.iloc[val_end:]

    return train, val, test

train, val, test = split_by_three_date(data, 'PurchDate')
print(len(train), len(val), len(test))

24327 24328 24328


# Preprocessing

In [4]:
num_features = train.select_dtypes(include="number").columns.tolist()

date_features = ["PurchMonth_cos", "PurchMonth_sin", "PurchDayOfWeek_cos", "PurchDayOfWeek_sin"]
# for date in date_features:
#     num_features.append(date)

cat_features = list(set(train.columns) - set(num_features) - set(date_features))
cat_features += ['BYRNO', 'VNZIP1'] # категориальные, которые были в числовых
num_features.remove('BYRNO')
num_features.remove('VNZIP1')
num_features.remove('IsBadBuy')
cat_features.remove("PurchDate")

target = 'IsBadBuy'

## Dates

In [5]:
train_dat = make_date_features(train, 'PurchDate')
val_dat = make_date_features(val, 'PurchDate')
test_dat = make_date_features(test, 'PurchDate')

train = pd.concat([train, train_dat], axis=1).drop(columns="PurchDate")
val = pd.concat([val, val_dat], axis=1).drop(columns="PurchDate")
test = pd.concat([test, test_dat], axis=1).drop(columns="PurchDate")

## Cat_features

In [6]:
count_enc = CountEncoder(cols = cat_features)

train_cat = count_enc.fit_transform(train[cat_features])
val_cat = count_enc.transform(val[cat_features])
test_cat = count_enc.transform(test[cat_features])

train_cat.head()

,WheelType,SubModel,VNST,Color,Auction,Model,Size,TopThreeAmericanName,Make,Transmission,Trim,Nationality,BYRNO,VNZIP1
32367,10622,284,2016,3432,14099,99,689,8098,2774,23646,4540,21168,1253,938
32384,12545,46,2016,5010,14099,309,2471,4629,4264,23646,265,21168,1096,938
32385,10622,1427,2016,5010,14099,250,9142,8098,4784,23646,3432,21168,1253,938
32386,12545,177,2016,4214,14099,55,2971,8441,5659,23646,2988,21168,1096,938
32387,12545,58,2016,1890,14099,955,9142,4629,4264,23646,265,21168,1096,938


## Num_features

In [7]:
drop_list = ['RefId', 'WheelTypeID', 'IsOnlineSale']
for ft in drop_list:
    num_features.remove(ft)

In [8]:
for col in num_features:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    val[col] = val[col].fillna(med)
    test[col] = test[col].fillna(med)
train_num = train[num_features]
val_num = val[num_features]
test_num = test[num_features]
train_num.head()

,VehYear,VehicleAge,VehOdo,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,MMRAcquisitonRetailCleanPrice,MMRCurrentAuctionAveragePrice,MMRCurrentAuctionCleanPrice,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,VehBCost,WarrantyCost
32367,2007,2,78541,7261.0,8857.0,8342.0,10066.0,8709.0,10331.0,9906.0,11657.0,6770.0,1389
32384,2005,4,37676,4409.0,5734.0,5262.0,6693.0,4908.0,5971.0,5801.0,6949.0,6160.0,941
32385,2004,5,71680,3098.0,4061.0,3846.0,4886.0,3397.0,4272.0,4169.0,5114.0,4250.0,1155
32386,2006,3,69456,8530.0,9883.0,9712.0,11174.0,9202.0,10794.0,10438.0,12158.0,8180.0,1703
32387,2004,5,66530,3094.0,4230.0,3842.0,5068.0,3369.0,4492.0,4139.0,5351.0,4900.0,825


In [9]:
X_train_cat = pd.concat([train_num, train[cat_features], train_dat], axis=1)
X_val_cat = pd.concat([val_num, val[cat_features], val_dat], axis=1)
X_test_cat = pd.concat([test_num, test[cat_features], test_dat], axis=1)
# X_train_cat.info()

In [10]:
for col in cat_features:
    X_train_cat[col] = X_train_cat[col].fillna("NaN").astype(str)
    X_val_cat[col] = X_val_cat[col].fillna("NaN").astype(str)
    X_test_cat[col] = X_test_cat[col].fillna("NaN").astype(str)

In [11]:
X_train = pd.concat([train_num, train_cat, train_dat], axis=1)
X_val = pd.concat([val_num, val_cat, val_dat], axis=1)
X_test = pd.concat([test_num, test_cat, test_dat], axis=1)
y_train, y_val, y_test = train[target], val[target], test[target]

In [12]:
print(X_train.shape, len(y_train))
print(X_val.shape, len(y_val))
print(X_test.shape, len(y_test))
# X_train.info()

(24327, 31) 24327
(24328, 31) 24328
(24328, 31) 24328


# Models implementation

#### DecisionTreeClassifier

In [13]:
SkTree = tree.DecisionTreeClassifier(max_depth = 7, min_samples_split = 50, min_samples_leaf = 10, class_weight=None)
SkTree.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",7
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",50
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curren

In [14]:
MyTree = MyDecisionTreeClassifier(weight_balance=None)
MyTree.fit(X_train, y_train)

In [15]:
print('SkTree')
pred = SkTree.predict(X_train)
proba = SkTree.predict_proba(X_train)

show_metrics(y_train, pred, proba[:, 1])
print('MyTree')

pred = MyTree.predict(X_train)
proba = MyTree.predict_proba(X_train)
my_score = gini(roc_auc_score(y_train, pred))

show_metrics(y_train, pred, proba)
print()

SkTree
Precision: 0.8626
Recall:    0.2359
F1:        0.3704
Gini:      0.5489
MyTree
Precision: 0.8305
Recall:    0.2491
F1:        0.3833
Gini:      0.5489



In [16]:
print('SkTree')
pred = SkTree.predict(X_val)
proba = SkTree.predict_proba(X_val)

show_metrics(y_val, pred, proba[:, 1])
print('MyTree')

pred = MyTree.predict(X_val)
proba = MyTree.predict_proba(X_val)
show_metrics(y_val, pred, proba)
print()

SkTree
Precision: 0.6971
Recall:    0.2386
F1:        0.3555
Gini:      0.4233
MyTree
Precision: 0.6675
Recall:    0.2487
F1:        0.3624
Gini:      0.4207



#### RandomForestClassifier

In [17]:
MyForest = MyRandomForestClassifier()
MyForest.fit(X_train, y_train)

In [18]:
pred = MyForest.predict(X_val, threshold=0.45)
proba = MyForest.predict_proba(X_val)
show_metrics(y_val, pred, proba)
print()

Precision: 0.8028
Recall:    0.2317
F1:        0.3596
Gini:      0.4736



## GDBTClassifier

In [19]:
MyGDBT = GDBTClassifier()
MyGDBT.fit(X_train, y_train)

In [ ]:
pred = MyGDBT.predict(X_val, threshold=0.5)
proba = MyGDBT.predict_proba(X_val)
my_score = gini(roc_auc_score(y_val, proba))
show_metrics(y_val, pred, proba)
proba.mean()

Precision: 0.8031
Recall:    0.2127
F1:        0.3363
Gini:      0.4712


np.float32(0.11483987)

## Raw libraries models

In [21]:
cat = CatBoostClassifier()
cat.fit(X_train, y_train, verbose=False)

CatBoostClassifier()

In [22]:
xgb = XGBClassifier()
xgb.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [23]:
lgbm = LGBMClassifier()
lgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 2794, number of negative: 21533
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000659 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3353
[LightGBM] [Info] Number of data points in the train set: 24327, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.114852 -> initscore=-2.042112
[LightGBM] [Info] Start training from score -2.042112


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [24]:
print('CATBOOST')
pred = cat.predict(X_val)
proba = cat.predict_proba(X_val)[:, 1]
my_score = gini(roc_auc_score(y_val, proba))
show_metrics(y_val, pred, proba)
print('XGB')
pred = xgb.predict(X_val)
proba = xgb.predict_proba(X_val)[:, 1]
my_score = gini(roc_auc_score(y_val, proba))
show_metrics(y_val, pred, proba)
print('LGBM')
pred = lgbm.predict(X_val)
proba = lgbm.predict_proba(X_val)[:, 1]
my_score = gini(roc_auc_score(y_val, proba))
show_metrics(y_val, pred, proba)
print()

CATBOOST
Precision: 0.7406
Recall:    0.2310
F1:        0.3522
Gini:      0.4565
XGB
Precision: 0.6389
Recall:    0.2355
F1:        0.3441
Gini:      0.4260
LGBM
Precision: 0.7238
Recall:    0.2402
F1:        0.3607
Gini:      0.4588



## Fine-tuning

### Catboost

In [25]:
cat_obj = make_CatBoost(X_train, y_train, X_val, y_val)
cat_study = optuna.create_study(direction="maximize")
cat_study.optimize(cat_obj, n_trials=50, show_progress_bar=True)

print("Best score:", cat_study.best_value)
print("Best params:", cat_study.best_params)

best_cat = CatBoostClassifier(**cat_study.best_params)
best_cat.fit(X_train, y_train, verbose=False)

Best trial: 19. Best value: 0.742779: 100%|██████████| 50/50 [01:05<00:00,  1.31s/it]


Best score: 0.7427794938856885
Best params: {'iterations': 500, 'learning_rate': 0.01827162741058002, 'depth': 6, 'min_data_in_leaf': 38, 'rsm': 0.6012939655642503}


CatBoostClassifier(depth=6, iterations=500, learning_rate=0.01827162741058002, min_data_in_leaf=38, rsm=0.6012939655642503)

In [26]:
pred = best_cat.predict(X_val)
proba = best_cat.predict_proba(X_val)[:,1]
cat_scores = show_metrics(y_val, pred, proba)

Precision: 0.7432
Recall:    0.2332
F1:        0.3551
Gini:      0.4840


### XGBoost

In [27]:
# for col in cat_features:
#     X_train_cat[col] = X_train_cat[col].astype("category")
#     X_val_cat[col] = X_val_cat[col].astype("category")

In [28]:
xgb_obj = make_XGBoost(X_train, y_train, X_val, y_val)
xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_obj, n_trials=50, show_progress_bar=True)

print("Best score:", xgb_study.best_value)
print("Best params:", xgb_study.best_params)

best_xgb = XGBClassifier(**xgb_study.best_params)
best_xgb.fit(X_train, y_train)

Best trial: 14. Best value: 0.74425: 100%|██████████| 50/50 [01:09<00:00,  1.39s/it] 


Best score: 0.7442502486750274
Best params: {'booster': 'gbtree', 'n_estimators': 250, 'learning_rate': 0.011780827109634098, 'max_depth': 6, 'min_child_weight': 17, 'colsample_bytree': 0.6547199082622437}


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,'gbtree'
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.6547199082622437
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRe

In [29]:
pred = best_xgb.predict(X_val)
proba = best_xgb.predict_proba(X_val)[:,1]
xgb_scores = show_metrics(y_val, pred, proba)

Precision: 0.7852
Recall:    0.2206
F1:        0.3444
Gini:      0.4864


### LightGBM

In [30]:
lgbm_obj = make_LightGBM(X_train, y_train, X_val, y_val)
lgbm_study = optuna.create_study(direction="maximize")
lgbm_study.optimize(lgbm_obj, n_trials=50, show_progress_bar=True)

print("Best score:", lgbm_study.best_value)
print("Best params:", lgbm_study.best_params)

best_lgbm = LGBMClassifier(**lgbm_study.best_params)
best_lgbm.fit(X_train, y_train)

Best trial: 42. Best value: 0.746369: 100%|██████████| 50/50 [01:13<00:00,  1.47s/it]


Best score: 0.7463693102520536
Best params: {'n_estimators': 450, 'learning_rate': 0.006392363885093605, 'max_depth': 7, 'min_child_samples': 22, 'feature_fraction': 0.708052782641129}


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,7
,learning_rate,0.006392363885093605
,n_estimators,450
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,22


In [31]:
pred = best_lgbm.predict(X_val)
proba = best_lgbm.predict_proba(X_val)[:,1]
lgbm_scores = show_metrics(y_val, pred, proba)

Precision: 0.7624
Recall:    0.2374
F1:        0.3620
Gini:      0.4915


In [32]:
print(f"cat: {cat_scores}\n xgb: {xgb_scores}\n lgbm: {lgbm_scores}")

cat: {'Precision': 0.743202416918429, 'Recall': 0.23324905183312264, 'F1': 0.3550637478951167, 'Gini': 0.4840487782965659}
 xgb: {'Precision': 0.7851518560179978, 'Recall': 0.22060682680151708, 'F1': 0.34443622008388847, 'Gini': 0.4864484952980528}
 lgbm: {'Precision': 0.7624365482233503, 'Recall': 0.23735777496839444, 'F1': 0.36201494335984574, 'Gini': 0.49151804904017293}


### MyGBDT

In [33]:
# my_obj = make_MyGBDT(X_train, y_train, X_val, y_val)
# my_study = optuna.create_study(direction="maximize")
# my_study.optimize(my_obj, n_trials=20, show_progress_bar=True)

# print("Best score:", my_study.best_value)
# print("Best params:", my_study.best_params)

# best_MyGDBT = GDBTClassifier(**my_study.best_params)
# best_MyGDBT.fit(X_train, y_train)

Best trial: 11. Best value: 0.741353: 100%|██████████| 20/20 [08:32<00:00, 25.61s/it]  
Best score: 0.7413527037420842  
Best params: {'lr': 0.13342455681511092, 'number_of_trees': 150, 'max_depth': 5, 'min_samples_split': 100, 'min_samples_leaf': 15, 'n_bins': 128}

In [34]:
my_obj = make_MyGBDT_2(X_train, y_train, X_val, y_val)
my_study = optuna.create_study(direction="maximize")
my_study.optimize(my_obj, n_trials=20, show_progress_bar=True)

print("Best score:", my_study.best_value)
print("Best params:", my_study.best_params)

best_MyGDBT = GDBTClassifier(**my_study.best_params)
best_MyGDBT.fit(X_train, y_train)

Best trial: 10. Best value: 0.743864: 100%|██████████| 20/20 [16:31<00:00, 49.58s/it]


Best score: 0.7438641094614546
Best params: {'lr': 0.19826683688406446, 'number_of_trees': 180, 'max_depth': 6, 'min_samples_split': 110, 'min_samples_leaf': 10}


In [35]:
pred = best_MyGDBT.predict(X_val)
proba = best_MyGDBT.predict_proba(X_val)
my_scores = show_metrics(y_val, pred, proba)

Precision: 0.7587
Recall:    0.2336
F1:        0.3572
Gini:      0.4853


### Categorial features by CatBoost

In [36]:
cat_obj = make_CatBoost(X_train_cat, y_train, X_val_cat, y_val, cat_cols=cat_features)
cat_study = optuna.create_study(direction="maximize")
cat_study.optimize(cat_obj, n_trials=50, show_progress_bar=True)

print("Best score:", cat_study.best_value)
print("Best params:", cat_study.best_params)

best_cat = CatBoostClassifier(**cat_study.best_params)
best_cat.fit(X_train_cat, y_train, cat_features=cat_features, verbose=False)

Best trial: 48. Best value: 0.744885: 100%|██████████| 50/50 [03:16<00:00,  3.93s/it]


Best score: 0.7448847866436362
Best params: {'iterations': 350, 'learning_rate': 0.03516399186749608, 'depth': 6, 'min_data_in_leaf': 4, 'rsm': 0.7130136358586295}


CatBoostClassifier(depth=6, iterations=350, learning_rate=0.03516399186749608, min_data_in_leaf=4, rsm=0.7130136358586295)

In [37]:
pred = best_cat.predict(X_val_cat)
proba = best_cat.predict_proba(X_val_cat)[:,1]
cat_scores_f = show_metrics(y_val, pred, proba)

Precision: 0.7500
Recall:    0.2446
F1:        0.3689
Gini:      0.4852


In [38]:
print(f"cat: {cat_scores}\n xgb: {xgb_scores}\n lgbm: {lgbm_scores}\n MyGBDT: {my_scores}\n cat_f: {cat_scores_f}")

cat: {'Precision': 0.743202416918429, 'Recall': 0.23324905183312264, 'F1': 0.3550637478951167, 'Gini': 0.4840487782965659}
 xgb: {'Precision': 0.7851518560179978, 'Recall': 0.22060682680151708, 'F1': 0.34443622008388847, 'Gini': 0.4864484952980528}
 lgbm: {'Precision': 0.7624365482233503, 'Recall': 0.23735777496839444, 'F1': 0.36201494335984574, 'Gini': 0.49151804904017293}
 MyGBDT: {'Precision': 0.7587268993839835, 'Recall': 0.23356510745891276, 'F1': 0.3571773803769937, 'Gini': 0.4853375517092329}
 cat_f: {'Precision': 0.75, 'Recall': 0.24462705436156765, 'F1': 0.36892278360343184, 'Gini': 0.48518088285787386}


## Проверка на test данных

In [39]:
print('xgb')
pred = best_xgb.predict(X_test)
proba = best_xgb.predict_proba(X_test)[:,1]
xgb_scores = show_metrics(y_test, pred, proba)
print('lgbm')
pred = best_lgbm.predict(X_test)
proba = best_lgbm.predict_proba(X_test)[:,1]
lgbest_lgbm_scores = show_metrics(y_test, pred, proba)
print('MyGBDT')
pred = best_MyGDBT.predict(X_test)
proba = best_MyGDBT.predict_proba(X_test)
my_scores = show_metrics(y_test, pred, proba)
print('cat')
pred = best_cat.predict(X_test_cat)
proba = best_cat.predict_proba(X_test_cat)[:,1]
cat_scores = show_metrics(y_test, pred, proba)
print()

xgb
Precision: 0.8742
Recall:    0.2164
F1:        0.3469
Gini:      0.4938
lgbm
Precision: 0.8392
Recall:    0.2353
F1:        0.3675
Gini:      0.4958
MyGBDT
Precision: 0.8658
Recall:    0.2372
F1:        0.3724
Gini:      0.4926
cat
Precision: 0.8566
Recall:    0.2416
F1:        0.3768
Gini:      0.5059



In [40]:
print(f"cat: {cat_scores}\n xgb: {xgb_scores}\n lgbm: {lgbm_scores}\n MyGBDT: {my_scores}\n cat_f: {cat_scores_f}")

cat: {'Precision': 0.8566392479435958, 'Recall': 0.2415506958250497, 'F1': 0.3768415611269062, 'Gini': 0.5059070728141708}
 xgb: {'Precision': 0.8741633199464525, 'Recall': 0.21636845593108017, 'F1': 0.3468791500664011, 'Gini': 0.49377657098236494}
 lgbm: {'Precision': 0.7624365482233503, 'Recall': 0.23735777496839444, 'F1': 0.36201494335984574, 'Gini': 0.49151804904017293}
 MyGBDT: {'Precision': 0.8657799274486094, 'Recall': 0.23724320742213387, 'F1': 0.37243172951885567, 'Gini': 0.4925703249609179}
 cat_f: {'Precision': 0.75, 'Recall': 0.24462705436156765, 'F1': 0.36892278360343184, 'Gini': 0.48518088285787386}


Как мы видим метрики на тестовых данных не упали, значит переобучения нет, регуляризация с размерами листов, бинами, и лр работает. Так что даже около 500 деревьев не переобучились.

## Extra Trees

In [41]:
Extra = ExtraTreesClassifier()
Extra.fit(X_train, y_train)

In [42]:
pred = Extra.predict(X_val)
proba = Extra.predict_proba(X_val)
show_metrics(y_val, pred, proba)
print()

Precision: 0.8358
Recall:    0.1239
F1:        0.2158
Gini:      0.4821



### сравнение одного дерева и ExtraTrees

In [43]:
pred = MyTree.predict(X_test)
proba = MyTree.predict_proba(X_test)
show_metrics(y_test, pred, proba)
print()

Precision: 0.7654
Recall:    0.2422
F1:        0.3680
Gini:      0.4309



In [44]:
pred = Extra.predict(X_test)
proba = Extra.predict_proba(X_test)
show_metrics(y_test, pred, proba)
print()

Precision: 0.9355
Recall:    0.1537
F1:        0.2641
Gini:      0.4911

